## Compare building heights: Global Atlas vs Cadastral
### Gap-fill missing Global Atlas heights using cadastral where possible


In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1.  DATA

- I have filter on buildings bigger then 5m2, I have not find it anywhere in the literature
- but I thougth since we use cadastral data, we wanna deduce shelter etc?

In [3]:
import os
os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

In [ ]:
bologna_boundary = gpd.read_file("data/bologna_boundary.gpkg", driver='GPKG').to_crs("EPSG:32632")
global_atlas = gpd.read_file("data/global_atlas_height.geojson").to_crs("EPSG:32632")
cadastral = gpd.read_file("data/catastral_map_with_heights.geojson").to_crs("EPSG:32632")

# 2. GLOBAL ATLAS 
global_atlas = gpd.clip(global_atlas, bologna_boundary)
global_atlas["area_m2"] = global_atlas.geometry.area
global_atlas["height_filtered"] = global_atlas["height"].where(global_atlas["area_m2"] > 5, 0)
print("GLOBAL ATLAS:")
print(f"Total buildings: {len(global_atlas):,}")
print(f"Height > 0: {(global_atlas['height_filtered'] > 0).sum():,}")
print(f"Zero/missing: {(global_atlas['height_filtered'] <= 0).sum():,}")

# 3. CADASTRAL 
cadastral_clipped = gpd.clip(cadastral, bologna_boundary)
cadastral_clipped["height"] = cadastral_clipped["altezza_gr"].fillna(cadastral_clipped["quota_gron"] - cadastral_clipped["quota_pied"])
cadastral_clipped["height_filtered"] = cadastral_clipped["height"].where(cadastral_clipped.geometry.area > 5, 0)
print("CADASTRAL:")
print(f"Total buildings: {len(cadastral_clipped):,}")
print(f"Height > 0: {(cadastral_clipped['height_filtered'] > 0).sum():,}")
print(f"Zero/missing: {(cadastral_clipped['height_filtered'] <= 0).sum():,}")

/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


GLOBAL ATLAS:
Total buildings: 53,533
Height > 0: 53,365
Zero/missing: 168
CADASTRAL:
Total buildings: 65,734
Height > 0: 54,372
Zero/missing: 11,362


## 2.  COMPARISON

join the geometries, compute overlap area and keep the best match.


In [ ]:

print("Comparing heights:") #global building atlas heights + location 
left = global_atlas[["geometry", "height_filtered"]].rename(columns={"height_filtered": "height_left"})

#cadastral height + location
right = cadastral_clipped[["geometry", "height_filtered"]].rename(columns={"height_filtered": "height_right"})

#compare by sjoin
joined = gpd.sjoin(
    left,
    right,
    how="inner",
    predicate="intersects"
)

# Compute overlap area
right_geoms = right.geometry

joined["overlap_area"] = joined.apply(
    lambda row: row.geometry.intersection(
        right_geoms.loc[row["index_right"]]
    ).area,
    axis=1
)

# Keep best match per building
joined = joined.sort_values("overlap_area", ascending=False)
joined = joined[~joined.index.duplicated(keep="first")]

Comparing heights:


Differences in the heights => compare the % of same of simular heights

In [ ]:

joined = joined[
    (joined["height_left"] > 0) &
    (joined["height_right"] > 0)
].copy()

#left = global buildings atlas, right = cadastral data
joined["height_diff"] = (
    joined["height_left"] - joined["height_right"]
).abs()

joined["signed_diff"] = (
    joined["height_left"] - joined["height_right"]
)

print("Absolute difference statistics:")
print(joined["height_diff"].describe(
    percentiles=[0.1,0.25,0.5,0.75,0.9,0.95]
))

Absolute difference statistics:
count    48831.000000
mean         4.464257
std          4.422187
min          0.000052
10%          0.584425
25%          1.499229
50%          3.265611
75%          5.932305
90%          9.612667
95%         13.054479
max         80.102523
Name: height_diff, dtype: float64


## Threshold sensitivity - differences accounted for different heigths thresholds


In [8]:

print("Threshold sensitivity:")
thresholds = [1, 2, 3, 5, 8, 10]

for t in thresholds:
    match_rate = (joined["height_diff"] <= t).mean()
    print(f"Tolerance ±{t} m → {match_rate*100:.1f}% match")

threshold_75 = joined["height_diff"].quantile(0.75) #75% of data threshold
median = joined["height_diff"].median()
mad = np.median(np.abs(joined["height_diff"] - median))
threshold_mad = median + 2 * mad
# Bias check - avg mean difference between the datasets
mean_bias = joined["signed_diff"].mean()

print(f"75th percentile threshold: {threshold_75:.2f} m")
print(f"Robust MAD threshold: {threshold_mad:.2f} m")
print(f"Mean signed difference (bias): {mean_bias:.2f} m")

# ==============================
no_match_left = left[~left.index.isin(joined.index)]
matched_right = joined["index_right"].unique()
no_match_right = right[~right.index.isin(matched_right)]

print(f"Global Atlas only: {len(no_match_left):,}")
print(f"Cadastral only: {len(no_match_right):,}")

Threshold sensitivity:
Tolerance ±1 m → 16.8% match
Tolerance ±2 m → 32.7% match
Tolerance ±3 m → 46.7% match
Tolerance ±5 m → 67.7% match
Tolerance ±8 m → 85.4% match
Tolerance ±10 m → 90.8% match
75th percentile threshold: 5.93 m
Robust MAD threshold: 7.37 m
Mean signed difference (bias): -2.51 m
Global Atlas only: 4,702
Cadastral only: 20,718


### GAP-FILL: CADASTRAL PRIMARY, GBA FALLBACK 
mean signed bias is -2.51 m = GBA understimates vs cadastral
=> gap-filling logic: use max() height from GBA to fill cadastral zeros = partially corrects for this downward bias


In [ ]:
#CADASTRAL AS BASE
print("Gap-filling: cadastral primary, GBA fallback:")
bologna_buildings_final = cadastral_clipped[["geometry", "height_filtered"]].copy()
bologna_buildings_final = bologna_buildings_final.rename(columns={"height_filtered": "height"})
bologna_buildings_final["source"] = "cadastral" #create a source col, for later usaeg -  when comparing sources

#1. CASE OF GAP FILLING: CADASTRAL: <=0
#identify cadastral buildings that need filling (height <= 0 or missing)
needs_fill_mask = bologna_buildings_final["height"] <= 0
needs_fill = bologna_buildings_final[needs_fill_mask].copy()
print(f"Cadastral buildings needing fill: {len(needs_fill):,}")

Gap-filling: cadastral primary, GBA fallback:
Cadastral buildings needing fill: 11,362


- 11,362 cadastral buildings exist but height ≤ 0 or null


## GAP FILLING

1. where cadastral height <= 0
2. where cadastral geometry is completely missing 



## cadastral as base

In [21]:
# ── STEP 1: BASE - cadastral as foundation ──────────────────────────────────
bologna_buildings_final = cadastral_clipped[["geometry", "height_filtered"]].copy()
bologna_buildings_final = bologna_buildings_final.rename(columns={"height_filtered": "height"})
bologna_buildings_final["source"] = "cadastral"
print(f"Base cadastral buildings: {len(bologna_buildings_final):,}")
print(f"Height > 0: {(bologna_buildings_final['height'] > 0).sum():,}")
print(f"Height <= 0 (needs fill): {(bologna_buildings_final['height'] <= 0).sum():,}")

Base cadastral buildings: 65,734
Height > 0: 54,372
Height <= 0 (needs fill): 11,362


### CASE A - fill missing heights from GBA 


In [22]:
needs_fill_mask = bologna_buildings_final["height"] <= 0
needs_fill = bologna_buildings_final[needs_fill_mask].copy()
needs_fill_buf = needs_fill.copy()
needs_fill_buf["geometry"] = needs_fill_buf.geometry.buffer(2)

gba_valid = global_atlas[global_atlas["height_filtered"] > 0][["geometry", "height_filtered"]].copy()

joined_fill = gpd.sjoin(
    needs_fill_buf[["geometry"]],
    gba_valid,
    how="left",
    predicate="intersects"
)

# One height per cadastral building - take first match (already deduplicated by best overlap earlier)
# One height per cadastral building - take first match
best_fill = joined_fill.dropna(subset=["height_filtered"]) \
                       .groupby(level=0)["height_filtered"].first()

bologna_buildings_final.loc[best_fill.index, "height"] = best_fill.values
bologna_buildings_final.loc[best_fill.index, "source"] = "cadastral_gba_fill"

filled_count = len(best_fill)
still_missing = (bologna_buildings_final["height"] <= 0).sum()
print(f"\nCase A - cadastral geometry, GBA height fill:")
print(f"  Successfully filled: {filled_count:,}")
print(f"  Still missing (no GBA match found): {still_missing:,}")



Case A - cadastral geometry, GBA height fill:
  Successfully filled: 7,296
  Still missing (no GBA match found): 4,066


### CASE B - append GBA buildings absent from cadastral entirely 


In [ ]:

matched_gba = gpd.sjoin(
    global_atlas[["geometry", "height_filtered"]],
    cadastral_clipped[["geometry"]],
    how="left",
    predicate="intersects"
)

# Get indices of GBA buildings that had NO cadastral match
unmatched_idx = matched_gba[matched_gba["index_right"].isna()].index.unique()
gba_no_cadastral = global_atlas.loc[unmatched_idx].copy()
gba_no_cadastral = gba_no_cadastral[gba_no_cadastral["height_filtered"] > 0]

gba_extra = gba_no_cadastral[["geometry", "height_filtered"]].rename(
    columns={"height_filtered": "height"}
)
gba_extra["source"] = "gba_only"

print(f"\nCase B - GBA buildings absent from cadastral entirely:")
print(f"  Appending: {len(gba_extra):,}")



Case B - GBA buildings absent from cadastral entirely:
  Appending: 2,388


In [24]:
bologna_buildings_final = pd.concat(
    [bologna_buildings_final, gba_extra],
    ignore_index=True
)
bologna_buildings_final = gpd.GeoDataFrame(
    bologna_buildings_final,
    geometry="geometry",
    crs="EPSG:32632"
)

#SUMMARY 
total = len(bologna_buildings_final)
print(f"\nFinal dataset:")
print(f"  Total buildings: {total:,}")
print(bologna_buildings_final.groupby("source")["height"].agg(["count", "mean"]).round(1))
print(f"  With height > 0: {(bologna_buildings_final['height'] > 0).sum():,}  ({(bologna_buildings_final['height'] > 0).mean()*100:.1f}%)")
print(f"  Still zero/missing: {(bologna_buildings_final['height'] <= 0).sum():,}")

# ── STEP 5: SAVE ─────────────────────────────────────────────────────────────
bologna_buildings_final.to_file("data/bologna_buildings_final.gpkg", driver="GPKG")
print(f"\nSaved: data/bologna_buildings_final.gpkg")


Final dataset:
  Total buildings: 70,510
                    count  mean
source                         
cadastral           58438  10.3
cadastral_gba_fill   7296   6.5
gba_only             4776   4.0
  With height > 0: 66,444  (94.2%)
  Still zero/missing: 4,066

Saved: data/bologna_buildings_final.gpkg
